FEEDBACK:
- There's not Product Category in the Data, therefore i couldnt complete the product ranking based on category, i did based on product instead
- The SCript didnt create the Catalogue i had to create it manually
- you mentioned to quarantine quantity <= 0 or a missing or empty customer_id to  however, the guide didnt mention removing from the Silver, so i kept it,. i could have remove from the silver b or filer out.

# BRONZE LAYER

In [0]:
%sql
Drop table tech_assessment.bronze.customer

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE tech_assessment.bronze.order AS
SELECT
  *,
  _metadata.file_name  AS source_file_path , 
  _metadata.file_modification_time as ingestion_time,
  current_timestamp()  AS processsing_time   
FROM STREAM read_files(
  '/Volumes/tech_assessment/bean_counter/landing/orders/',
  format => 'csv',
  header => true
)


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE tech_assessment.bronze.customer AS
SELECT
  *,
  _metadata.file_name  AS source_file_path , 
   _metadata.file_modification_time as ingestion_time,
  current_timestamp()  AS processsing_time  
FROM STREAM read_files(
  '/Volumes/tech_assessment/bean_counter/landing/customers/',
  format => 'json',
  header => true
)


# SILVER LAYER

In [0]:


df_order_silver=spark.sql( """
select 
distinct
o.order_id,
o.customer_id,
o.store_id,
o.product_id,
o.quantity,
coalesce(o.unit_price,p.unit_price) as unit_price,
o.order_ts,
COALESCE(
  try_to_date(o.order_ts, 'yyyy-MM-dd HH:mm:ss'),
  try_to_date(o.order_ts, 'yyyy-MM-dd HH:mm'),
  try_to_date(o.order_ts, 'dd/MM/yyyy HH:mm:ss'),
  try_to_date(o.order_ts, 'dd/MM/yyyy HH:mm')
) AS order_date,

date_format(
  COALESCE(
    try_to_timestamp(o.order_ts, 'yyyy-MM-dd HH:mm:ss'),
    try_to_timestamp(o.order_ts, 'yyyy-MM-dd HH:mm'),
    try_to_timestamp(o.order_ts, 'dd/MM/yyyy HH:mm:ss'),
    try_to_timestamp(o.order_ts, 'dd/MM/yyyy HH:mm'),
    try_to_timestamp(o.order_ts, 'H:mm:ss'),
    try_to_timestamp(o.order_ts, 'H:mm')
  ),
  'HH:mm:ss'
) AS order_time,
o.source_file_path,
o.ingestion_timestamp

from tech_assessment.bronze.order o
LEFT JOIN read_files(
  '/Volumes/tech_assessment/bean_counter/landing/products.csv',
  format => 'csv',
  header => true
) p
ON o.product_id = p.product_id
""")

(df_order_silver.write
.format("delta") 
.mode("append") 
.saveAsTable("tech_assessment.silver.silver_order")
)

In [0]:
%sql
CREATE OR REPLACE TABLE tech_assessment.silver.silver_orders_quarantine
AS
SELECT
    *,
    CASE
        WHEN quantity <= 0 THEN 'Invalid_quantity (<= 0)'
        WHEN customer_id IS NULL OR TRIM(customer_id) = '' THEN 'Missing_customer_id'
    END AS rejection_reason
FROM tech_assessment.bronze.order
WHERE quantity <= 0
   OR customer_id IS NULL
   OR TRIM(customer_id) = '';


In [0]:

spark.sql(
""" 

WITH CUSTOMER AS (
SELECT 
    city,
    customer_id,
    email,
    marketing_opt_in,
    signup_date,
    updated_at AS  latest_updated_at,
    source_file_path,
    ingestion_timestamp,
    row_number() over (partition by customer_id order by updated_at desc) current_row
 FROM tech_assessment.bronze.customer
)

SELECT 
    city,
    customer_id,
    email,
    marketing_opt_in,
    signup_date,
    latest_updated_at,
    source_file_path,
    ingestion_timestamp
FROM CUSTOMER WHERE current_row = 1
 """
).createOrReplaceTempView("customer_source")



## CREATING CUSTOMER TABLE

In [0]:
%sql
create Table if not exists tech_assessment.silver.silver_customer (
city varchar(50),
customer_id varchar(50),
email varchar(100),
marketing_opt_in varchar(50),
signup_date date,
latest_updated_at timestamp,
source_file_path varchar(200),
ingestion_timestamp timestamp
)

In [0]:
%sql
MERGE INTO tech_assessment.silver.silver_customer target
using customer_source
on customer_source.customer_id = target.customer_id
when matched then update SET *
when not matched then INSERT *

# GOLD LAYER

In [0]:

spark.sql(
    """
create or replace table tech_assessment.gold.gold_daily_sales
    using delta
SELECT 
    store_id,
    order_date,
    count(distinct order_id) as distinct_orders,
    round(sum(quantity * unit_price),2)as revenue,
    sum(quantity) as unit_sold
FROM tech_assessment.silver.silver_order
where quantity > 0 and customer_id is not null 
group by store_id,order_date
order by order_date, store_id
    
    """
)


In [0]:

spark.sql( 
        
"""
        
create or replace table tech_assessment.gold.gold_product_ranking  
using delta    
    WITH Product_revenue AS (
    SELECT
    product_id,
    round(sum(quantity * unit_price),2) as revenue

from  tech_assessment.silver.silver_order
where quantity > 0 and customer_id is not null 
group by product_id
)

select 
    product_id,
    revenue,
    dense_rank() over (order by revenue desc) as product_rank
from Product_revenue;
"""
)



